In [0]:
%pip install -U -qqqq mlflow databricks-openai databricks-agents databricks-langchain
dbutils.library.restartPython()

### Simple LLM Call

In [0]:
from databricks_langchain import ChatDatabricks

chat_model = ChatDatabricks(
    # endpoint="databricks-dbrx-instruct",
    endpoint="databricks-claude-3-7-sonnet",
    temperature=0.1,
    max_tokens=256,
    # See https://python.langchain.com/api_reference/community/chat_models/langchain_community.chat_models.databricks.ChatDatabricks.html for other supported parameters
)

In [0]:
chat_model.invoke("What is MLflow?")

In [0]:
# You can also pass a list of messages
messages = [
    ("system", "You are a chatbot that can answer questions about Databricks."),
    ("user", "What is Databricks Model Serving?"),
]
chat_model.invoke(messages)

In [0]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a chatbot that can answer questions about {topic}.",
        ),
        ("user", "{question}"),
    ]
)

chain = prompt | chat_model
chain.invoke(
    {
        "topic": "Databricks",
        "question": "What is Unity Catalog?",
    }
)

In [0]:
for chunk in chat_model.stream("How are you?"):
    # print(chunk.content, end="|")
    print(chunk.content)

#### LLM with RAG and index search

#### Langchain LLM chain

### Agents

#### Simple Agent

In [0]:
from langchain.agents import create_agent
agent = create_agent(model="openai:gpt-5",system_prompt=”you are...”,tools=tools)


### Agent calling python code

In [0]:
import json
import mlflow
from databricks.sdk import WorkspaceClient
from databricks_openai import UCFunctionToolkit, DatabricksFunctionClient
# Import MLflow utilities for converting from chat completions to Responses API format
from mlflow.types.responses import output_to_responses_items_stream, create_function_call_output_item

# Enable automatic tracing for easier debugging
mlflow.openai.autolog()

# Get an OpenAI client configured to connect to Databricks model serving endpoints
openai_client = WorkspaceClient().serving_endpoints.get_open_ai_client()

# Load Databricks built-in tools (Python code interpreter)
client = DatabricksFunctionClient()
builtin_tools = UCFunctionToolkit(function_names=["system.ai.python_exec"], client=client).tools
for tool in builtin_tools:
  del tool["function"]["strict"]

def call_tool(tool_name, parameters):
  if tool_name == "system__ai__python_exec":
    return DatabricksFunctionClient().execute_function("system.ai.python_exec", parameters=parameters).value
  raise ValueError(f"Unknown tool: {tool_name}")

def call_llm(prompt):
  for chunk in openai_client.chat.completions.create(
    model="databricks-claude-3-7-sonnet",
    messages=[{"role": "user", "content": prompt}],
    tools=builtin_tools,
    stream=True
  ):
    yield chunk.to_dict()

def run_agent(prompt):
  """
  Send a user prompt to the LLM, and yield LLM + tool call responses
  The LLM is allowed to call the code interpreter tool if needed, to respond to the user
  """
  # Convert output into Responses API-compatible events
  for chunk in output_to_responses_items_stream(call_llm(prompt)):
    yield chunk.model_dump(exclude_none=True)
  # If the model executed a tool, call it and yield the tool call output in Responses API format
  if chunk.item.get('type') == 'function_call':
    tool_name = chunk.item["name"]
    tool_args = json.loads(chunk.item["arguments"])
    tool_result = call_tool(tool_name, tool_args)
    yield {"type": "response.output_item.done", "item": create_function_call_output_item(call_id=chunk.item["call_id"], output=tool_result)}

In [0]:
for output_chunk in run_agent("What is the square root of 429?"):
  print(output_chunk)


###Langchain Python UDF tools

In [0]:
from langchain.agents import create_agent

# from langchain.agents import initialize_agent
from databricks_langchain import ChatDatabricks

# 1. Define the function to modify a UC table
@tool
def python_udf_summarize(x, y):
    """Returns summary of two numeric parameters"""
    return x + y

# Another tool example
@tool
def python_udf_multiple(x, y):
    """Returns multiplication of two numeric parameters"""
    return x * y


tools=[python_udf_summarize,python_udf_multiple]
# 3. Add the tool to your agent
llm = ChatDatabricks(endpoint="databricks-claude-sonnet-4", temperature=0.1)
agent = create_agent(
    tools=tools,
    model=llm
)

# Example usage
agent.invoke({"messages": "Summarize 5 and 7 multplied to 10"})

###Langchain SQL UC UDF tools

In [0]:
%sql
USE CATALOG learn_adb_fikrat;
USE bronze;
CREATE OR REPLACE FUNCTION python_udf_summarize(x DOUBLE, y DOUBLE)
RETURNS DOUBLE
LANGUAGE PYTHON
COMMENT 'Returns summary of two numeric parameters'
AS
$$
return x + y
$$;


In [0]:
%sql
CREATE OR REPLACE FUNCTION sql_udf_accident_stats()
RETURNS TABLE (borough STRING, NUMBER_OF_PERSONS_INJURED INT,NUMBER_OF_PERSONS_KILLED INT)
LANGUAGE SQL
COMMENT 'Returns number of killed and injured people in vehicle accidents'
RETURN 
SELECT borough, NUMBER_OF_PERSONS_INJURED,NUMBER_OF_PERSONS_KILLED  FROM learn_adb_fikrat.silver.vehicle_accidents_cleansed_stream3;

In [0]:
from langchain.agents import create_agent
from langchain.tools import tool
from databricks_langchain import UCFunctionToolkit

func_name = f"learn_adb_fikrat.bronze.sql_udf_accident_stats"
toolkit = UCFunctionToolkit(function_names=[func_name])
# Define a tool that calls the SQL UDF

tools = toolkit.tools

agent = create_agent(
    tools=tools,
    model=llm
)

# Example usage
agent.invoke({"messages": "What are the accident stats for Manhattan in 2023?"})

Different example (To be completed)

In [0]:
# The code below is SQL, not Python. To fix the syntax and convert to Python Spark DataFrame API:

# Set the catalog and schema
spark.sql("USE CATALOG learn_adb_fikrat")
spark.sql("USE bronze")

# Create or replace a Python UDF in Databricks (registering via SQL is correct, but here's a Python version for Spark)
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

@udf(DoubleType())
def python_udf_summarize(x, y):
    """Returns summary of two numeric parameters"""
    return x + y

# Register the UDF for SQL usage
spark.udf.register("python_udf_summarize", python_udf_summarize)